In [63]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

1. # IMPORT UNCLEANED DATA

In [64]:
csv_uncleaned = pd.read_csv(r"C:\Users\JB\OneDrive\Desktop\synthetic_mobile_money_transaction_dataset_uncleaned.csv")

# DROPPING THE "STEPS" FROM THE COLUMN

In [65]:
# Check if 'step' exists, then drop it
if 'step' in csv_uncleaned.columns:
    csv_cleaned = csv_uncleaned.drop(columns=['step'])


# Verify
print("Columns after dropping 'step':", csv_cleaned.columns)


Columns after dropping 'step': Index(['transactionType', 'amount', 'initiator', 'oldBalInitiator',
       'newBalInitiator', 'recipient', 'oldBalRecipient', 'newBalRecipient',
       'isFraud'],
      dtype='object')


# ADDING REGIONS

In [66]:


# List of all 16 regions in Ghana
regions = [
    "Greater Accra", "Ashanti", "Eastern", "Central", "Western",
    "Bono", "Bono East", "Ahafo", "Volta", "Oti",
    "Northern", "North East", "Savannah", "Upper East", "Upper West", "Western North"
]

# Raw weights (approximate popularity for transactions)
weights = [
    25, 20, 10, 5, 5,
    4, 3, 3, 5, 3,
    5, 2, 2, 2, 1, 1
]

# Normalize weights so they sum to 1
probabilities = np.array(weights) / sum(weights)

# Add 'region' column with weighted random selection
csv_cleaned['region'] = np.random.choice(regions, size=len(csv_cleaned), p=probabilities)

csv_cleaned['country']='Ghana'
print(csv_cleaned[['region','country']].head())

# Check distribution
print(csv_cleaned['region'].value_counts(normalize=True))






          region country
0           Bono   Ghana
1  Greater Accra   Ghana
2  Greater Accra   Ghana
3          Volta   Ghana
4        Eastern   Ghana
region
Greater Accra    0.260845
Ashanti          0.207736
Eastern          0.103774
Western          0.052282
Central          0.052163
Volta            0.052089
Northern         0.052043
Bono             0.041700
Ahafo            0.031298
Bono East        0.031234
Oti              0.031198
Savannah         0.021070
Upper East       0.020966
North East       0.020688
Western North    0.010477
Upper West       0.010437
Name: proportion, dtype: float64


# ADDING TIME , GENDER AND AGES


In [72]:



# 1. Add 'timestamp' column (random between 2023-01-01 and 2025-01-31)

start_datetime = datetime(2023, 1, 1)
end_datetime = datetime(2025, 1, 31)
total_seconds = int((end_datetime - start_datetime).total_seconds())

random_seconds = np.random.randint(0, total_seconds, size=len(csv_cleaned))
csv_cleaned['timestamp'] = [start_datetime + timedelta(seconds=int(s)) for s in random_seconds]

# 2. Add 'age_group' column (weighted by transaction frequency)

age_groups = ["18-24", "25-34", "35-44", "45-54", "55-64", "60+"]
# Most transactions come from younger groups
age_weights = [30, 25, 15, 12, 11, 7]  
age_probabilities = np.array(age_weights) / sum(age_weights)

csv_cleaned['age_group'] = np.random.choice(age_groups, size=len(csv_cleaned), p=age_probabilities)

# Check results

print(csv_cleaned[['age_group', 'timestamp']].head())
print("\nAge group distribution:\n", csv_cleaned['age_group'].value_counts(normalize=True))



# Add 'gender' column
# ----------------------------
genders = ["Male", "Female"]
# Assign probabilities (e.g., 55% male, 45% female)
gender_probabilities = [0.60, 0.40]

csv_cleaned['gender'] = np.random.choice(genders, size=len(csv_cleaned), p=gender_probabilities)

# ----------------------------
# Check distribution
# ----------------------------
print("\nGender distribution:\n", csv_cleaned['gender'].value_counts(normalize=True))
print(csv_cleaned[['gender']].head())



  age_group           timestamp
0     45-54 2024-06-30 21:19:10
1     25-34 2023-12-22 00:27:20
2     18-24 2024-12-15 06:42:31
3     18-24 2023-10-17 17:13:54
4       60+ 2023-08-02 10:30:55

Age group distribution:
 age_group
18-24    0.299463
25-34    0.249902
35-44    0.150067
45-54    0.120415
55-64    0.109873
60+      0.070279
Name: proportion, dtype: float64

Gender distribution:
 gender
Male      0.599601
Female    0.400399
Name: proportion, dtype: float64
   gender
0  Female
1    Male
2  Female
3  Female
4    Male


# DATAOVERVIEW

In [73]:



# 1. Basic info

print("===== DATA SHAPE =====")
print(csv_cleaned.shape)

print("\n===== COLUMNS & DATA TYPES =====")
print(csv_cleaned.dtypes)


# 2. Missing values

print("\n===== MISSING VALUES =====")
print(csv_cleaned.isnull().sum())


# 3. Summary statistics for numeric columns

print("\n===== NUMERIC SUMMARY =====")
print(csv_cleaned.describe())


# 4. Summary for categorical columns

categorical_cols = csv_cleaned.select_dtypes(include='object').columns
print("\n===== CATEGORICAL SUMMARY =====")
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(csv_cleaned[col].value_counts())
    print(csv_cleaned[col].value_counts(normalize=True))  # percentage

# 5. Example: Distribution of key columns

if 'age_group' in csv_cleaned.columns:
    print("\nAge group distribution:\n", csv_cleaned['age_group'].value_counts(normalize=True))

if 'region' in csv_cleaned.columns:
    print("\nRegion distribution:\n", csv_cleaned['region'].value_counts(normalize=True))

if 'transactionType' in csv_cleaned.columns:
    print("\nTransaction type distribution:\n", csv_cleaned['transactionType'].value_counts(normalize=True))

if 'amount' in csv_cleaned.columns:
    print("\nTransaction amount stats:\n", csv_cleaned['amount'].describe())


===== DATA SHAPE =====
(1720181, 14)

===== COLUMNS & DATA TYPES =====
transactionType            object
amount                    float64
initiator                   int64
oldBalInitiator           float64
newBalInitiator           float64
recipient                  object
oldBalRecipient           float64
newBalRecipient           float64
isFraud                     int64
region                     object
country                    object
timestamp          datetime64[ns]
age_group                  object
gender                     object
dtype: object

===== MISSING VALUES =====
transactionType    0
amount             0
initiator          0
oldBalInitiator    0
newBalInitiator    0
recipient          0
oldBalRecipient    0
newBalRecipient    0
isFraud            0
region             0
country            0
timestamp          0
age_group          0
gender             0
dtype: int64

===== NUMERIC SUMMARY =====
             amount     initiator  oldBalInitiator  newBalInitiator  \
coun

# SAVING THE DATASET 

In [74]:
csv_cleaned.to_csv(r"C:\Users\JB\OneDrive\Desktop\Joseph_cleaned.csv", index=False)